# **Introduzione a Python asyncio: concetti base**

## **Il problema della concorrenza**

Si immagini di dover implementare un programma che scarichi 10 file da Internet: con un approccio sequenziale tradizionale, il programma scaricherebbe un file alla volta: mentre aspetta che arrivino i dati del primo file, la CPU resta sostanzialmente inattiva. È come fare la fila alla posta: mentre si aspetta il proprio turno, non si può fare altro.

Come noto, una possibile soluzione a questo problema è quella di usare **threads concorrenti**, ognuno dei quali scarica un singolo file. Questo approccio risolve certamente il problema, velocizzando il download complessivo dei file, ma "sovraccarica" il sistema operativo (SO) che deve occuparsi di gestire l'esecuzione concorrente dei threads.

L'approccio **asincrono** risolve il problema con una filosofia differente, permettendo ad un programma di "fare altre cose" mentre aspetta operazioni lente (I/O su rete, disco, ecc.). Non è multithreading, ma **programmazione asincrona cooperativa**: c'è un unico processo in esecuzione, in grado di passare da un compito all'altro ogni volta che si verifica una qualche attesa.

Questo *notebook* introduce gli elementi base del modulo `asyncio`, utilizzato per la programmazione asincrona in Python.

---


## 1. **Event Loop: il cuore pulsante**

In una applicazione asincrona, l'**event Loop** (o *ciclo degli eventi*) è il motore che coordina tutto: può essere pensato come un direttore d'orchestra che decide chi suona e quando.

#### Come funziona:

```
┌──────────────────────────────────────────────────────────────┐
│                      EVENT LOOP                              │
│  1. Controlla la coda dei task in attesa di essere eseguiti  │
│  2. Esegue il primo task pronto                              │
│  3. Se il task è in attesa (await), passa al prossimo        │
│  4. Torna periodicamente ai task che stavano aspettando      │
└──────────────────────────────────────────────────────────────┘
```

Il **concetto chiave** è che l'event loop è **single-threaded** (*mono-thread*): esegue una parte di codice (*task*) alla volta, ma può passare rapidamente da un task all'altro quando questi sono in attesa, mantenendo un unico flusso di esecuzione.

---


## 2. **Coroutines: funzioni cooperative** (che possono "mettersi in pausa")

Una **coroutine** è una funzione speciale che può sospendere la propria esecuzione ogni volta che è necessario (ad esempio quando effettua una operazione lenta), restituendo il controllo all'event loop. Successivamente, quando l'operazione lenta si è conclusa e l'event loop la chiama nuovamente in causa, la coroutine riprende l'esecuzione dal punto immediatamente successivo a quello in cui si è sospesa.

Una *coroutine* viene definita con **`async def`**: `async` è dunque la parola chiave che trasforma una funzione in una *coroutine*.

#### Esempio base:

```python
import asyncio

async def scarica_file(nome):
    '''coroutine che simula il download di un file'''
    print(f"Inizio download di {nome}")
    await asyncio.sleep(2)                  #simula il tempo necessario per il download
    print(f"Completato download di {nome}")
    return f"Contenuto di {nome}"
```

Una *coroutine*:
- si definisce con `async def`;
- può usare `await` al suo interno (e <u>dovrebbe farlo almeno una volta</u>!);
- se viene chiamata, **non viene eseguita subito**, ma restituisce un oggetto coroutine;
- deve essere eseguita dall'event loop.

In [ ]:
import asyncio

async def scarica_file(nome):
    '''coroutine che simula il download di un file'''
    print(f"Inizio download di {nome}")
    await asyncio.sleep(2)                  #simula il tempo necessario per il download
    print(f"Completato download di {nome}")
    return f"Contenuto di {nome}"

#la chiamata della coroutine NON esegue la funzione!
coro = scarica_file("documento.pdf")
print(type(coro))                           #coro è un oggetto coroutine

#per eseguire la coroutine nel notebook, occorre effettuare la chiamata preceduta da 'await'
result = await scarica_file("documento.pdf")
print(result)

**Osservazione importante**: per eseguire la coroutine nel notebook si è utilizzato lo *statement* `result = await scarica_file("documento.pdf")`. Questo è dovuto al fatto che il notebook possiede già un proprio loop.

In un programma indipendente occorre invece istanziare e avviare esplicitamente l'event loop mediante `asyncio.run()`:
```python
asyncio.run(scarica_file("documento.pdf"))
```

## 3. **await: cedere il controllo**

`await` è la parola chiave che, all'interno di una coroutine, deve essere messa davanti alla chiamata di una "operazione lenta" e sta proprio a significare: "questa operazione potrebbe richiedere tempo, perciò io mi sospendo così nel frattempo il resto del programma può fare altro".

#### Meccanismo di funzionamento:

```python
async def esempio():
    print("1. Inizio")
    risultato = await operazione_lenta()    #← qui la coroutine si sospende e cede il controllo all'event loop
    print("3. Riprendo dopo l'attesa")
    return risultato
```

**Quando il programma incontra `await`**:
1. la coroutine si "congela" in quel punto e restituisce il controllo all'event loop;
2. l'event loop può eseguire altre coroutines;
3. quando `operazione_lenta()` è completata, l'event loop risveglia la coroutine originale;
4. l'esecuzione riprende dalla riga successiva all'`await`.

#### Regole importanti:
- `await` può essere usato **solo dentro funzioni async**;
- `await` può essere usato solo su oggetti "awaitable" (coroutine, Task, Future).

### **Threads vs Coroutines**
A questo punto, è importante puntualizzare ancora una volta la differenza tra l'approccio multithreaded e l'approccio asincrono:
- in un processo *multithreaded*, i threads sono flussi indipendenti la cui esecuzione è gestita dal SO secondo le proprie regole di scheduling;
- in un processo *asincrono*, invece, il flusso di esecuzione è unico e orchestrato dall'event loop, che riprende il controllo ogni volta che una coroutine si sospende per mandarne in esecuzione un altra.

Le coroutine sono definite **cooperative** proprio perchè decidono autonomamente di sospendersi ogni volta che sanno che stanno per imbattersi in un "tempo morto", così da permettere al resto del programma di proseguire. Questa è un'altra differenza fondamentale: tramite l'utilizzo di `await`, infatti, è il programma stesso a stabilire quando è il momento giusto per sospendere un task, senza demandare questa decisione al SO, che utilizza criteri differenti.

## 4. **Tasks: coroutines "in esecuzione"**

Un **task** è un *wrapper* che "avvolge" una coroutine e la schedula per l'esecuzione nell'event loop. Se vengono schedulati più task contemporaneamente, la loro esecuzione è concorrente (nel senso di asyncio).

#### Differenza coroutine vs task:

```python
# coroutine: è solo una "promessa" di esecuzione
coro = scarica_file("file1.txt")

# task: è schedulato nell'event loop ed eseguito concorrentemente
task = asyncio.create_task(scarica_file("file1.txt"))
```


#### Esempio di esecuzione concorrente di tre tasks:

In [ ]:
async def scarica_file(nome):
    print(f"Inizio download di {nome}")
    await asyncio.sleep(2)
    print(f"Completato download di {nome}")
    return f"Contenuto di {nome}"

async def main():
    #crea tre task e li schedula nell'event loop per l'esecuzione concorrente
    task1 = asyncio.create_task(scarica_file("file1.txt"))
    task2 = asyncio.create_task(scarica_file("file2.txt"))
    task3 = asyncio.create_task(scarica_file("file3.txt"))

    #(avvia e) aspetta che tutti i task finiscano
    risultati = await asyncio.gather(task1, task2, task3)
    print(risultati)

await main()    #esecuzione della coroutine principale nel notebook

**Timeline dell'esecuzione**:

```
Tempo →
0s:  [Task1 inizia] [Task2 inizia] [Task3 inizia]
     ↓               ↓               ↓
     await sleep     await sleep     await sleep
     (cede)          (cede)          (cede)
     
2s:  [Task1 riprende e completa]
     [Task2 riprende e completa]
     [Task3 riprende e completa]
```

---

**Osservazione importante**: la "struttura" evidenziata nell'esempio precedente è tipica di una applicazione asincrona:
1. si definiscono le coroutines corrispondenti ai tasks concorrenti che devono essere eseguiti;
2. si definisce una coroutine principale (`main()`) che crea e gestisce i tasks corrispondenti alle coroutines del punto 1.;
3. si avvia l'event loop con il task corrispondente alla coroutine principale:
    - `await main()` nel *notebook*;
    - `asyncio.run(main())` in un programma indipendente.

## 5. **Interazione tra i componenti: un esempio completo**

In [ ]:
import asyncio
import time

#coroutine 1 ---> task 1
async def prepara_caffe(nome):
    print(f"{nome}: Inizio preparazione caffè")
    await asyncio.sleep(3)              #macchina del caffè lavora
    print(f"{nome}: Caffè pronto!")
    return f"Caffè {nome}"

#coroutine 2 ---> task 2
async def tosta_pane(nome):
    print(f"{nome}: Inizio tostatura pane")
    await asyncio.sleep(2)              #tostapane lavora
    print(f"{nome}: Pane tostato!")
    return f"Toast {nome}"

#coroutine principale --> task principale
async def prepara_colazione():
    print("=== Inizio preparazione colazione  ===")
    inizio = time.time()

    #avvio concorrente di più operazioni
    task_caffe = asyncio.create_task(prepara_caffe("Mario"))
    task_pane = asyncio.create_task(tosta_pane("Mario"))

    #attesa termine operazioni avviate
    caffe, pane = await asyncio.gather(task_caffe, task_pane)

    fine = time.time()
    print(f"=== Colazione pronta in {fine - inizio:.1f} secondi ===")
    print(f"Hai: {caffe} e {pane}")

#avvio dell'event loop che esegue il task principale
#asyncio.run(prepara_colazione()) --> in un programma si dovrebbe usare questo
await prepara_colazione() #esecuzione nel notebook

#### Analisi del flusso:
1. **`asyncio.run()`** crea e avvia l'event loop;
2. event loop esegue `prepara_colazione()`;
3. in `prepara_colazione()` vengono creati due **tasks** che partono "insieme";
4. entrambi i tasks arrivano a `await asyncio.sleep()` e **cedono il controllo**;
5. event loop tiene traccia del tempo;
6. quando un task è pronto (sleep finito), lo risveglia;
7. **asyncio.gather()** aspetta che tutti i task completino;
8. tempo totale di esecuzione: 3 secondi (non 5!), perché le operazioni sono sovrapposte.
---

## **Schema riassuntivo delle interazioni**

```
┌──────────────────────────────────────────────┐
│              EVENT LOOP                      │
│  (gestisce tutti i task in un solo thread)   │
└────────────┬─────────────────────────────────┘
             │
             │ schedula
             ↓
    ┌────────────────┐
    │   TASK 1       │──→ wrappa una COROUTINE
    └────────────────┘        │
                              │ async def funzione():
                              │     ...
                              ↓     await altra_coro()
                         (si sospende qui)
                              │
                              └─→ restituisce controllo
                                  all'event loop
             │
             │ nel frattempo esegue
             ↓
    ┌────────────────┐
    │   TASK 2       │──→ altra COROUTINE
    └────────────────┘
```

---

## **Confronto: sincrono vs asincrono**

#### Codice sincrono (tempo totale: 6 secondi):

In [ ]:
import time

def operazione_lenta(n):
    time.sleep(2)
    return f"Risultato {n}"

inizio = time.time()
risultato1 = operazione_lenta(1)
risultato2 = operazione_lenta(2)
risultato3 = operazione_lenta(3)
print(f"Tempo: {time.time() - inizio:.1f}s")  # ~6.0s

#### Codice asincrono (tempo totale: 2 secondi):

In [ ]:
import asyncio
import time

async def operazione_lenta(n):
    await asyncio.sleep(2)
    return f"Risultato {n}"

async def main():
    inizio = time.time()
    risultati = await asyncio.gather(
        operazione_lenta(1),
        operazione_lenta(2),
        operazione_lenta(3)
    )
    print(f"Tempo: {time.time() - inizio:.1f}s")  # ~2.0s

# asyncio.run(main())
await main()

---

## **Conclusione: punti chiave da ricordare**

1. **event Loop**: il coordinatore che gestisce l'esecuzione dei tasks;
2. **coroutine**: funzione che può auto-sospendersi (definita con `async def`),;
3. **await**: punto in cui la coroutine cede il controllo;
4. **task**: coroutine schedulata per l'esecuzione concorrente;
5. **asyncio.run()**: avvia l'event loop e esegue la coroutine principale;
6. **asyncio.gather()**: esegue più coroutine/task concorrentemente e attende tutti i risultati;

#### Quando usare asyncio:
- ✅ operazioni I/O-bound (rete, database, file)
- ✅ quando serve gestire molte connessioni simultanee
- ❌ **non** per operazioni CPU-intensive (calcoli pesanti) → usare multiprocessing

---